The following is an example of how to utilize our Sen1Floods11 dataset for training a FCNN. In this example, we train and validate on hand-labeled chips of flood events. However, our dataset includes several other options that are detailed in the README. To replace the dataset, as outlined further below, simply replace the train, test, and validation split csv's, and download the corresponding dataset.

Authenticate Google Cloud Platform. Note that to run this code, you must connect your notebook runtime to a GPU.

1

In [151]:
!pip install -q rasterio segmentation-models-pytorch

2

In [152]:
import csv
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

import segmentation_models_pytorch as smp

from google.colab import drive

drive.mount("/content/drive")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("设备:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

DATA_ROOT = Path("/content/sen1floods11")
S1_DIR = DATA_ROOT / "S1"
LABEL_DIR = DATA_ROOT / "Labels"
SPLIT_DIR = DATA_ROOT / "splits"

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Sen1Floods11_UNet"
)
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
RESULT_DIR = PROJECT_DIR / "results"

for directory in [
    S1_DIR,
    LABEL_DIR,
    SPLIT_DIR,
    CHECKPOINT_DIR,
    RESULT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("项目目录:", PROJECT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
设备: cuda
GPU: Tesla T4
项目目录: /content/drive/MyDrive/Sen1Floods11_UNet


3

In [153]:
!gsutil cp \
  gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_train_data.csv \
  /content/sen1floods11/splits/

!gsutil cp \
  gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_valid_data.csv \
  /content/sen1floods11/splits/

!gsutil cp \
  gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_test_data.csv \
  /content/sen1floods11/splits/

!gsutil -m rsync -r \
  gs://sen1floods11/v1.1/data/flood_events/HandLabeled/S1Hand \
  /content/sen1floods11/S1

!gsutil -m rsync -r \
  gs://sen1floods11/v1.1/data/flood_events/HandLabeled/LabelHand \
  /content/sen1floods11/Labels

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_train_data.csv...
/ [1 files][ 13.3 KiB/ 13.3 KiB]                                                
Operation completed over 1 objects/13.3 KiB.                                     
Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://sen1floods11/v1.1/splits/flood_handlabeled/flood_valid_data.csv...
/ [1 files][  4.7 KiB/  4.7 KiB]                                                
Operation completed over 1 objects/4.7 KiB.                                   

In [154]:
print("S1数量:", len(list(S1_DIR.glob("*.tif"))))
print("标签数量:", len(list(LABEL_DIR.glob("*.tif"))))
print("划分文件:", list(SPLIT_DIR.glob("*.csv")))

S1数量: 446
标签数量: 446
划分文件: [PosixPath('/content/sen1floods11/splits/flood_train_data.csv'), PosixPath('/content/sen1floods11/splits/flood_valid_data.csv'), PosixPath('/content/sen1floods11/splits/flood_test_data.csv')]


4

In [155]:
class Sen1Floods11Dataset(Dataset):
    def __init__(
        self,
        split_csv,
        s1_dir,
        label_dir,
        training=False,
        crop_size=256,
        water_crop_probability=0.6,
        min_water_pixels=100,
    ):
        self.s1_dir = Path(s1_dir)
        self.label_dir = Path(label_dir)
        self.training = training
        self.crop_size = crop_size
        self.water_crop_probability = water_crop_probability
        self.min_water_pixels = min_water_pixels

        self.items = []

        with open(
            split_csv,
            newline="",
            encoding="utf-8",
        ) as file:
            for row in csv.reader(file):
                if len(row) < 2:
                    continue

                s1_name = Path(row[0]).name
                label_name = Path(row[1]).name

                s1_path = self.s1_dir / s1_name
                label_path = self.label_dir / label_name

                if not s1_path.exists():
                    raise FileNotFoundError(s1_path)

                if not label_path.exists():
                    raise FileNotFoundError(label_path)

                self.items.append(
                    (s1_name, label_name)
                )

        print(
            f"{Path(split_csv).name}: "
            f"{len(self.items)}个样本"
        )

    def __len__(self):
        return len(self.items)

    def _water_aware_crop(self, image, mask):
        _, height, width = image.shape
        size = self.crop_size

        if height < size or width < size:
            raise ValueError(
                f"影像尺寸{(height, width)}"
                f"小于裁剪尺寸{size}"
            )

        require_water = (
            random.random()
            < self.water_crop_probability
        )

        selected_top = 0
        selected_left = 0

        for _ in range(10):
            top = random.randint(
                0,
                height - size,
            )
            left = random.randint(
                0,
                width - size,
            )

            crop_mask = mask[
                top:top + size,
                left:left + size,
            ]

            selected_top = top
            selected_left = left

            if not require_water:
                break

            water_pixels = (
                crop_mask == 1
            ).sum().item()

            if water_pixels >= self.min_water_pixels:
                break

        image = image[
            :,
            selected_top:selected_top + size,
            selected_left:selected_left + size,
        ]

        mask = mask[
            selected_top:selected_top + size,
            selected_left:selected_left + size,
        ]

        return image, mask

    def __getitem__(self, index):
        s1_name, label_name = self.items[index]

        with rasterio.open(
            self.s1_dir / s1_name
        ) as src:
            if src.count < 2:
                raise ValueError(
                    f"{s1_name}没有VV/VH两个波段"
                )

            # Band 1=VV，Band 2=VH
            image = src.read(
                [1, 2]
            ).astype(np.float32)

        with rasterio.open(
            self.label_dir / label_name
        ) as src:
            mask = src.read(1).astype(np.int64)

        # 官方预处理：不添加强度噪声
        image = np.nan_to_num(
            image,
            nan=0.0,
            posinf=1.0,
            neginf=-50.0,
        ).astype(np.float32)

        image = np.clip(
            image,
            -50.0,
            1.0,
        )

        image = (
            image + 50.0
        ) / 51.0

        # -1表示无效或未标注
        mask[mask == -1] = 255

        image = torch.from_numpy(
            image
        ).float()

        mask = torch.from_numpy(
            mask
        ).long()

        if self.training:
            image, mask = self._water_aware_crop(
                image,
                mask,
            )

            if random.random() < 0.5:
                image = torch.flip(
                    image,
                    dims=[2],
                )
                mask = torch.flip(
                    mask,
                    dims=[1],
                )

            if random.random() < 0.5:
                image = torch.flip(
                    image,
                    dims=[1],
                )
                mask = torch.flip(
                    mask,
                    dims=[0],
                )

            rotation_k = random.randint(0, 3)

            if rotation_k > 0:
                image = torch.rot90(
                    image,
                    rotation_k,
                    dims=[1, 2],
                )

                mask = torch.rot90(
                    mask,
                    rotation_k,
                    dims=[0, 1],
                )

        mean = torch.tensor(
            [0.6851, 0.5235],
            dtype=torch.float32,
        )[:, None, None]

        std = torch.tensor(
            [0.0820, 0.1102],
            dtype=torch.float32,
        )[:, None, None]

        image = (
            image - mean
        ) / std

        return (
            image.contiguous(),
            mask.contiguous(),
        )

5

In [156]:
train_dataset = Sen1Floods11Dataset(
    SPLIT_DIR / "flood_train_data.csv",
    S1_DIR,
    LABEL_DIR,
    training=True,
)

valid_dataset = Sen1Floods11Dataset(
    SPLIT_DIR / "flood_valid_data.csv",
    S1_DIR,
    LABEL_DIR,
    training=False,
)

test_dataset = Sen1Floods11Dataset(
    SPLIT_DIR / "flood_test_data.csv",
    S1_DIR,
    LABEL_DIR,
    training=False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

flood_train_data.csv: 252个样本
flood_valid_data.csv: 89个样本
flood_test_data.csv: 90个样本


In [157]:
images, masks = next(iter(train_loader))

print("影像:", images.shape, images.dtype)
print("标签:", masks.shape, masks.dtype)
print("标签值:", torch.unique(masks))
print("存在NaN:", torch.isnan(images).any().item())

影像: torch.Size([16, 2, 256, 256]) torch.float32
标签: torch.Size([16, 256, 256]) torch.int64
标签值: tensor([  0,   1, 255])
存在NaN: False


6

In [158]:
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=2,
    classes=2,
    activation=None,
).to(device)

model.eval()

with torch.no_grad():
    test_output = model(
        images[:2].to(device)
    )

print("输出尺寸:", test_output.shape)
print(
    "输出包含NaN:",
    torch.isnan(test_output).any().item(),
)

输出尺寸: torch.Size([2, 2, 256, 256])
输出包含NaN: False


7

In [159]:
ce_loss = nn.CrossEntropyLoss(
    weight=torch.tensor(
        [1.0, 8.0],
        device=device,
    ),
    ignore_index=255,
)

def combined_loss(logits, masks):
    ce = ce_loss(logits, masks)

    valid = masks != 255
    true_water = (masks == 1).float()

    water_probability = torch.softmax(
        logits,
        dim=1,
    )[:, 1]

    water_probability = water_probability[valid]
    true_water = true_water[valid]

    intersection = (
        water_probability * true_water
    ).sum()

    denominator = (
        water_probability.sum()
        + true_water.sum()
    )

    dice = (
        2.0 * intersection + 1e-6
    ) / (
        denominator + 1e-6
    )

    dice_loss = 1.0 - dice

    return (
        0.5 * ce
        + 0.5 * dice_loss
    )

8

In [160]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=max(len(train_loader) * 10, 1),
    T_mult=2,
    eta_min=1e-7,
)

def run_epoch(
    model,
    loader,
    training,
    optimizer=None,
    scheduler=None,
):
    model.train(training)

    total_loss = 0.0
    intersection = 0
    union = 0
    correct = 0
    valid_pixels = 0

    for batch_images, batch_masks in tqdm(
        loader,
        leave=False,
    ):
        batch_images = batch_images.to(
            device,
            non_blocking=True,
        )

        batch_masks = batch_masks.to(
            device,
            non_blocking=True,
        )

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(batch_images)

            loss = combined_loss(
                logits,
                batch_masks,
            )

            if training:
                loss.backward()
                optimizer.step()

                if scheduler is not None:
                    scheduler.step()

        prediction = logits.argmax(dim=1)
        valid = batch_masks != 255

        predicted_water = (
            prediction == 1
        ) & valid

        true_water = (
            batch_masks == 1
        ) & valid

        intersection += (
            predicted_water & true_water
        ).sum().item()

        union += (
            predicted_water | true_water
        ).sum().item()

        correct += (
            (prediction == batch_masks)
            & valid
        ).sum().item()

        valid_pixels += valid.sum().item()
        total_loss += loss.item()

    return {
        "loss": (
            total_loss / max(len(loader), 1)
        ),
        "water_iou": (
            intersection / max(union, 1)
        ),
        "accuracy": (
            correct / max(valid_pixels, 1)
        ),
    }

9

In [ ]:
BEST_MODEL_PATH = (
    CHECKPOINT_DIR
    / "best_unet_resnet34_stable.pt"
)

MAX_EPOCHS = 50
PATIENCE = 15

best_iou = -1.0
epochs_without_improvement = 0
history = []

for epoch in range(MAX_EPOCHS):
    train_metrics = run_epoch(
        model,
        train_loader,
        training=True,
        optimizer=optimizer,
        scheduler=scheduler,
    )

    valid_metrics = run_epoch(
        model,
        valid_loader,
        training=False,
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_metrics["loss"],
        "train_iou": train_metrics["water_iou"],
        "train_accuracy": train_metrics["accuracy"],
        "valid_loss": valid_metrics["loss"],
        "valid_iou": valid_metrics["water_iou"],
        "valid_accuracy": valid_metrics["accuracy"],
    })

    print(
        f"epoch={epoch + 1} "
        f"train_loss={train_metrics['loss']:.4f} "
        f"train_iou={train_metrics['water_iou']:.4f} "
        f"valid_loss={valid_metrics['loss']:.4f} "
        f"valid_iou={valid_metrics['water_iou']:.4f} "
        f"valid_acc={valid_metrics['accuracy']:.4f}"
    )

    if valid_metrics["water_iou"] > best_iou:
        best_iou = valid_metrics["water_iou"]
        epochs_without_improvement = 0

        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_valid_iou": best_iou,
                "architecture": "unet_resnet34",
                "in_channels": 2,
                "classes": 2,
                "mean": [0.6851, 0.5235],
                "std": [0.0820, 0.1102],
            },
            BEST_MODEL_PATH,
        )

        print(
            f"保存最佳模型，"
            f"valid IoU={best_iou:.4f}"
        )

    else:
        epochs_without_improvement += 1

    pd.DataFrame(history).to_csv(
        RESULT_DIR
        / "stable_training_history.csv",
        index=False,
    )

    if epochs_without_improvement >= PATIENCE:
        print(
            f"连续{PATIENCE}个epoch未提升，"
            "提前停止"
        )
        break

  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/45 [00:00<?, ?it/s]

epoch=1 train_loss=0.6158 train_iou=0.2987 valid_loss=0.6571 valid_iou=0.2451 valid_acc=0.6767
保存最佳模型，valid IoU=0.2451


  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/45 [00:00<?, ?it/s]

...

epoch=22 train_loss=0.3740 train_iou=0.4987 valid_loss=0.3523 valid_iou=0.6136 valid_acc=0.9380


  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/45 [00:00<?, ?it/s]

epoch=23 train_loss=0.3474 train_iou=0.5465 valid_loss=0.3837 valid_iou=0.6347 valid_acc=0.9484
保存最佳模型，valid IoU=0.6347


  0%|          | 0/16 [00:00<?, ?it/s]

10

In [ ]:
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(device)
model.eval()

print("最佳epoch:", checkpoint["epoch"])
print(
    "保存的最佳验证IoU:",
    checkpoint["best_valid_iou"],
)

verification_metrics = run_epoch(
    model,
    valid_loader,
    training=False,
)

print("重新验证:", verification_metrics)

11

In [ ]:
@torch.no_grad()
def scan_thresholds(
    model,
    loader,
    thresholds,
):
    model.eval()

    stats = {
        float(t): {
            "tp": 0,
            "fp": 0,
            "fn": 0,
            "scene_ious": [],
            "water_scene_ious": [],
        }
        for t in thresholds
    }

    for images, masks in tqdm(loader):
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)

        probabilities = torch.softmax(
            logits,
            dim=1,
        )[:, 1]

        valid = masks != 255
        true_water = (masks == 1) & valid

        for threshold in thresholds:
            threshold = float(threshold)

            predicted_water = (
                probabilities >= threshold
            ) & valid

            tp = (
                predicted_water & true_water
            ).sum().item()

            fp = (
                predicted_water
                & ~true_water
                & valid
            ).sum().item()

            fn = (
                ~predicted_water
                & true_water
                & valid
            ).sum().item()

            stats[threshold]["tp"] += tp
            stats[threshold]["fp"] += fp
            stats[threshold]["fn"] += fn

            for i in range(len(masks)):
                sample_true = true_water[i]
                sample_pred = predicted_water[i]

                intersection = (
                    sample_true & sample_pred
                ).sum().item()

                union = (
                    sample_true | sample_pred
                ).sum().item()

                if union > 0:
                    iou = intersection / union

                    stats[threshold][
                        "scene_ious"
                    ].append(iou)

                    if sample_true.sum().item() > 0:
                        stats[threshold][
                            "water_scene_ious"
                        ].append(iou)

    rows = []

    for threshold, values in stats.items():
        tp = values["tp"]
        fp = values["fp"]
        fn = values["fn"]

        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)

        f1 = (
            2 * precision * recall
            / max(precision + recall, 1e-8)
        )

        global_iou = (
            tp / max(tp + fp + fn, 1)
        )

        rows.append({
            "threshold": threshold,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "global_iou": global_iou,
            "macro_iou": float(
                np.mean(values["scene_ious"])
            ),
            "water_macro_iou": float(
                np.mean(
                    values["water_scene_ious"]
                )
            ),
        })

    return pd.DataFrame(rows)

In [ ]:
thresholds = np.arange(
    0.10,
    0.96,
    0.05,
)

threshold_results = scan_thresholds(
    model,
    valid_loader,
    thresholds,
)

display(
    threshold_results.sort_values(
        "water_macro_iou",
        ascending=False,
    )
)

In [ ]:
best_threshold_index = threshold_results[
    "water_macro_iou"
].idxmax()

best_threshold_row = threshold_results.loc[
    best_threshold_index
]

BEST_THRESHOLD = float(
    best_threshold_row["threshold"]
)

print("最佳阈值:", BEST_THRESHOLD)
print(best_threshold_row)

threshold_results.to_csv(
    RESULT_DIR
    / "stable_threshold_results.csv",
    index=False,
)

12

In [ ]:
@torch.no_grad()
def evaluate_dataset(
    model,
    loader,
    threshold,
):
    model.eval()

    tp = fp = fn = tn = 0
    scene_ious = []
    water_scene_ious = []

    for images, masks in tqdm(loader):
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)

        probabilities = torch.softmax(
            logits,
            dim=1,
        )[:, 1]

        predicted_water = (
            probabilities >= threshold
        )

        valid = masks != 255
        predicted_water &= valid

        true_water = (masks == 1) & valid
        true_land = (masks == 0) & valid

        tp += (
            predicted_water & true_water
        ).sum().item()

        fp += (
            predicted_water & true_land
        ).sum().item()

        fn += (
            ~predicted_water
            & true_water
            & valid
        ).sum().item()

        tn += (
            ~predicted_water
            & true_land
            & valid
        ).sum().item()

        for i in range(len(masks)):
            sample_true = true_water[i]
            sample_pred = predicted_water[i]

            intersection = (
                sample_true & sample_pred
            ).sum().item()

            union = (
                sample_true | sample_pred
            ).sum().item()

            if union > 0:
                iou = intersection / union
                scene_ious.append(iou)

                if sample_true.sum().item() > 0:
                    water_scene_ious.append(iou)

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)

    f1 = (
        2 * precision * recall
        / max(precision + recall, 1e-8)
    )

    global_iou = (
        tp / max(tp + fp + fn, 1)
    )

    accuracy = (
        (tp + tn)
        / max(tp + fp + fn + tn, 1)
    )

    scene_ious = np.asarray(scene_ious)
    water_scene_ious = np.asarray(
        water_scene_ious
    )

    return {
        "threshold": float(threshold),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "global_iou": global_iou,
        "accuracy": accuracy,
        "macro_iou": float(
            scene_ious.mean()
        ),
        "median_iou": float(
            np.median(scene_ious)
        ),
        "water_macro_iou": float(
            water_scene_ious.mean()
        ),
        "iou_below_02": float(
            np.mean(scene_ious < 0.2)
        ),
        "iou_at_least_05": float(
            np.mean(scene_ious >= 0.5)
        ),
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "tn": tn,
    }

In [ ]:
test_metrics = evaluate_dataset(
    model,
    test_loader,
    BEST_THRESHOLD,
)

for name, value in test_metrics.items():
    if isinstance(value, float):
        print(f"{name}: {value:.4f}")
    else:
        print(f"{name}: {value}")

13

In [ ]:
FINAL_MODEL_PATH = (
    CHECKPOINT_DIR
    / "final_unet_resnet34_stable.pt"
)

final_checkpoint = checkpoint.copy()
final_checkpoint["threshold"] = BEST_THRESHOLD
final_checkpoint["test_metrics"] = test_metrics

torch.save(
    final_checkpoint,
    FINAL_MODEL_PATH,
)

with open(
    RESULT_DIR / "stable_test_metrics.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        test_metrics,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("最终模型:", FINAL_MODEL_PATH)

14

In [ ]:
SAMPLE_INDEX = 0

image, mask = test_dataset[SAMPLE_INDEX]

with torch.no_grad():
    logits = model(
        image.unsqueeze(0).to(device)
    )

    probability = torch.softmax(
        logits,
        dim=1,
    )[0, 1].cpu()

prediction = probability >= BEST_THRESHOLD

vv = (
    image[0] * 0.0820 + 0.6851
).numpy()

low, high = np.percentile(vv, [2, 98])

vv_display = np.clip(
    (vv - low) / max(high - low, 1e-8),
    0,
    1,
)

mask_display = np.ma.masked_where(
    mask.numpy() == 255,
    mask.numpy(),
)

fig, axes = plt.subplots(
    1,
    4,
    figsize=(20, 5),
)

axes[0].imshow(
    vv_display,
    cmap="gray",
)
axes[0].set_title("Sentinel-1 VV")

axes[1].imshow(
    mask_display,
    cmap="Blues",
    vmin=0,
    vmax=1,
)
axes[1].set_title("Ground truth")

axes[2].imshow(
    prediction.numpy(),
    cmap="Blues",
    vmin=0,
    vmax=1,
)
axes[2].set_title(
    f"Prediction, threshold={BEST_THRESHOLD:.2f}"
)

probability_plot = axes[3].imshow(
    probability.numpy(),
    cmap="viridis",
    vmin=0,
    vmax=1,
)

axes[3].set_title("Water probability")
plt.colorbar(
    probability_plot,
    ax=axes[3],
)

for axis in axes:
    axis.axis("off")

plt.tight_layout()
plt.show()